# Burundi VBR — Sanity Check

Pick one OU with **good** data quality (declared ≈ verified ≈ validated) and one with **bad** quality (large discrepancies), then verify that the simulation selects them for verification as expected.

In [ ]:
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(200)

QUALITY_PATH = "cleaned_quantity_data_ext_1805.csv"
SIM_PATH = "VBR_verification_information.csv"

quality = pl.read_csv(QUALITY_PATH, infer_schema_length=10_000)
sim = pl.read_csv(SIM_PATH, infer_schema_length=10_000)

QUALITY_MONTHS = [202509, 202510, 202511, 202512, 202601, 202602]
quality = quality.filter(pl.col("month").is_in(QUALITY_MONTHS))
sim = sim.filter(pl.col("qtrisk") == "ecartdecver")

print(f"Quality rows: {quality.shape[0]:,}  cols: {quality.shape[1]}")
print(f"Sim rows:     {sim.shape[0]:,}  cols: {sim.shape[1]}")

Quality rows: 27,112  cols: 43
Sim rows:     8,604  cols: 54


## 1. Data quality per OU

Summarise each OU by its average `taux_validation` (validated / declared, closer to 1 = better) and `ecart_dec_val`. Only rows where verification actually occurred (`ver` and `val` non-null) are included, and only OUs with ≥ 10 such rows. Sorted best → worst.

In [ ]:
ou_quality = (
    quality.filter(
        pl.col("dec").is_not_null()
        & (pl.col("dec") > 0)
        & pl.col("ver").is_not_null()
        & pl.col("val").is_not_null()
    )
    .group_by("ou")
    .agg(
        pl.col("level_5_name").first(),
        pl.col("taux_validation").mean().alias("mean_taux_validation"),
        pl.col("ecart_dec_val").mean().alias("mean_ecart_dec_val"),
        pl.len().alias("n_rows"),
    )
    .filter(pl.col("n_rows") >= 10)
    .sort("mean_taux_validation", descending=True)  # best (≈1) at top, worst at bottom
)

display(ou_quality)

ou,level_5_name,mean_taux_validation,mean_ecart_dec_val,n_rows
str,str,f64,f64,u32
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",1.0,0.0,12
"""VQR6XZWtvJs""","""CDS Mwumba""",1.0,0.0,35
"""lr9EwOFjk8H""","""SWAA Ruyigi""",1.0,0.0125,10
"""owoA4p2yc04""","""CDS Kiryama""",1.0,0.0,45
"""HzOjo12RnMR""","""CDS Prison Gitega""",1.0,0.0,12
"""lr9VmpDZg1v""","""CDS Condi""",1.0,0.000572,53
"""NrDbZMw60v7""","""CDS Bururi""",1.0,0.0,30
"""eV3zuFGTDmc""","""CDS Horezo""",1.0,0.0,54
"""hHpkd4lhvhj""","""CDS Kajabure""",0.999874,0.000835,51


## 2. Select representative OUs

- **Good OU**: highest `mean_taux_validation` → declared and validated almost identical.
- **Bad OU**: lowest `mean_taux_validation` → largest discrepancy.

In [15]:
good_ou = ou_quality.head(1)["ou"][0]
bad_ou = ou_quality.tail(1)["ou"][0]

print("GOOD OU")
display(ou_quality.head(1))

print("BAD OU")
display(ou_quality.tail(1))

GOOD OU


ou,level_5_name,mean_taux_validation,mean_ecart_dec_val,n_rows
str,str,f64,f64,u32
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",1.0,0.0,12


BAD OU


ou,level_5_name,mean_taux_validation,mean_ecart_dec_val,n_rows
str,str,f64,f64,u32
"""TdO8ymL2ZFj""","""CMC Dunamai""",0.499349,0.548114,12


## 3. Service-level detail

Inspect the raw declared / verified / validated quantities per service to confirm the selection makes intuitive sense.

In [ ]:
DETAIL_COLS = [
    "level_5_name",
    "month",
    "service",
    "dec",
    "ver",
    "val",
    "ecart_dec_ver",
    "ecart_dec_val",
    "taux_validation",
]

for label, ou in [("GOOD OU", good_ou), ("BAD OU", bad_ou)]:
    print(f"\n=== {label}: {ou} ===")
    display(
        quality.filter(pl.col("ou") == ou)
        .select(DETAIL_COLS)
        .unique(subset=["month", "service"], keep="first")
        .sort(["month", "service"])
    )


=== GOOD OU: cJkvG3w5ZyD ===


level_5_name,month,service,dec,ver,val,ecart_dec_ver,ecart_dec_val,taux_validation
str,i64,str,f64,f64,f64,f64,f64,f64
"""CDS Prison Bururi""",202509,"""Nouvelle Consultation Curative…",2.0,2.0,2.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202510,"""Nouvelle Consultation Curative…",3.0,3.0,3.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202511,"""Nouvelle Consultation Curative…",4.0,4.0,4.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202511,"""Nouvelle Consultation Curative…",251.0,251.0,251.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202512,"""Nouvelle Consultation Curative…",3.0,3.0,3.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202512,"""Nouvelle Consultation Curative…",238.0,238.0,238.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202601,"""Nouvelle Consultation Curative…",2.0,2.0,2.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202601,"""Nouvelle Consultation Curative…",257.0,257.0,257.0,0.0,0.0,1.0
"""CDS Prison Bururi""",202601,"""Nouvelle consultation curative…",2.0,2.0,2.0,0.0,0.0,1.0



=== BAD OU: TdO8ymL2ZFj ===


level_5_name,month,service,dec,ver,val,ecart_dec_ver,ecart_dec_val,taux_validation
str,i64,str,f64,f64,f64,f64,f64,f64
"""CMC Dunamai""",202509,"""Consultation prénatale recentr…",41.0,37.0,0.0,0.108108,1.108108,0.0
"""CMC Dunamai""",202509,"""FP: Tot. NA + AA des méthodes …",202.0,203.0,203.0,0.004926,0.004926,1.0
"""CMC Dunamai""",202510,"""Consultation prénatale recentr…",48.0,32.0,0.0,0.5,1.5,0.0
"""CMC Dunamai""",202510,"""FP: Tot. NA + AA des méthodes …",224.0,224.0,224.0,0.0,0.0,1.0
"""CMC Dunamai""",202511,"""Consultation prénatale recentr…",41.0,41.0,0.0,0.0,1.0,0.0
"""CMC Dunamai""",202511,"""FP: Tot. NA + AA des méthodes …",128.0,128.0,127.0,0.0,0.0078125,0.9921875
"""CMC Dunamai""",202512,"""Consultation prénatale recentr…",33.0,33.0,0.0,0.0,1.0,0.0
"""CMC Dunamai""",202512,"""FP: Tot. NA + AA des méthodes …",132.0,132.0,132.0,0.0,0.0,1.0
"""CMC Dunamai""",202601,"""Consultation prénatale recentr…",31.0,31.0,0.0,0.0,1.0,0.0


## 4. Simulation results

Look up how the VBR simulation classified each OU:
- `categorie_risque` — overall risk category (high / moderate / low / uneligible)
- `bool_verified` — 1 if selected for verification
- `category` — outcome label

In [17]:
SIM_COLS = [
    "ou_id",
    "level_5_name",
    "periode",
    "qtrisk",
    "taux_validation",
    "median_weighted_ecart",
    "categorie_risque_weighted_ecart",
    "categorie_risque_dec_ver_ecart",
    "categorie_risque_gain_verification",
    "categorie_risque",
    "bool_verified",
    "category",
]

for label, ou in [("GOOD OU", good_ou), ("BAD OU", bad_ou)]:
    print(f"\n=== {label}: {ou} ===")
    result = sim.filter(pl.col("ou_id") == ou).select(SIM_COLS)
    if result.is_empty():
        print("  !! OU not found in simulation results")
    else:
        display(result)


=== GOOD OU: cJkvG3w5ZyD ===


ou_id,level_5_name,periode,qtrisk,taux_validation,median_weighted_ecart,categorie_risque_weighted_ecart,categorie_risque_dec_ver_ecart,categorie_risque_gain_verification,categorie_risque,bool_verified,category
str,str,i64,str,f64,f64,str,str,str,str,i64,str
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202603,"""ecartdecver""",1.0,0.0,"""low""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202603,"""ecartdecver""",1.0,0.0,"""low""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202603,"""ecartdecver""",1.0,0.0,"""low""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202604,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202604,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202604,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",1,"""Not declared services for the …"
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202605,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202605,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""cJkvG3w5ZyD""","""CDS Prison Bururi""",202605,"""ecartdecver""",1.0,0.0,"""low""","""low""","""high""","""low""",0,"""Not declared services for the …"



=== BAD OU: TdO8ymL2ZFj ===


ou_id,level_5_name,periode,qtrisk,taux_validation,median_weighted_ecart,categorie_risque_weighted_ecart,categorie_risque_dec_ver_ecart,categorie_risque_gain_verification,categorie_risque,bool_verified,category
str,str,i64,str,f64,f64,str,str,str,str,i64,str
"""TdO8ymL2ZFj""","""CMC Dunamai""",202603,"""ecartdecver""",0.5,0.304348,"""moderate_1""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""TdO8ymL2ZFj""","""CMC Dunamai""",202603,"""ecartdecver""",0.5,0.304348,"""high""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""TdO8ymL2ZFj""","""CMC Dunamai""",202603,"""ecartdecver""",0.5,0.304348,"""moderate_1""","""low""","""moderate_1""","""low""",0,"""Non bénéfique & Non vérifié"""
"""TdO8ymL2ZFj""","""CMC Dunamai""",202604,"""ecartdecver""",0.5,0.3,"""moderate_1""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""TdO8ymL2ZFj""","""CMC Dunamai""",202604,"""ecartdecver""",0.5,0.3,"""moderate_1""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""TdO8ymL2ZFj""","""CMC Dunamai""",202604,"""ecartdecver""",0.5,0.3,"""high""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""TdO8ymL2ZFj""","""CMC Dunamai""",202605,"""ecartdecver""",0.5,0.3,"""high""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""TdO8ymL2ZFj""","""CMC Dunamai""",202605,"""ecartdecver""",0.5,0.3,"""moderate_1""","""low""","""high""","""low""",0,"""Not declared services for the …"
"""TdO8ymL2ZFj""","""CMC Dunamai""",202605,"""ecartdecver""",0.5,0.3,"""moderate_1""","""low""","""high""","""low""",0,"""Not declared services for the …"


## 5. Summary

Side-by-side comparison: quality metrics (left) vs simulation outcome (right).

Expected: good OU → low risk, `bool_verified = 0`; bad OU → high/moderate risk, `bool_verified = 1`.

In [18]:
sim_agg = (
    sim.filter(pl.col("ou_id").is_in([good_ou, bad_ou]))
    .group_by("ou_id")
    .agg(
        pl.col("categorie_risque").first(),
        pl.col("bool_verified").first(),
        pl.col("category").first(),
    )
)

summary = (
    ou_quality.filter(pl.col("ou").is_in([good_ou, bad_ou]))
    .rename({"ou": "ou_id"})
    .join(sim_agg, on="ou_id", how="left")
    .with_columns(
        pl.when(pl.col("ou_id") == good_ou)
        .then(pl.lit("GOOD"))
        .otherwise(pl.lit("BAD"))
        .alias("type")
    )
    .select(
        [
            "type",
            "ou_id",
            "level_5_name",
            "mean_taux_validation",
            "mean_ecart_dec_val",
            "categorie_risque",
            "bool_verified",
            "category",
        ]
    )
    .sort("type", descending=True)  # GOOD first
)

display(summary)

type,ou_id,level_5_name,mean_taux_validation,mean_ecart_dec_val,categorie_risque,bool_verified,category
str,str,str,f64,f64,str,i64,str
"""GOOD""","""cJkvG3w5ZyD""","""CDS Prison Bururi""",1.0,0.0,"""low""",0,"""Non bénéfique & Non vérifié"""
"""BAD""","""TdO8ymL2ZFj""","""CMC Dunamai""",0.499349,0.548114,"""low""",0,"""Non bénéfique & Non vérifié"""
